In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
from jaxcmr.summarize import (
    calculate_aic_weights,
    generate_t_p_matrices,
    summarize_parameters,
    winner_comparison_matrix,
    raw_winner_comparison_matrix,
    pairwise_aic_differences,
    pairwise_median_aic_differences,
    bayesian_model_selection,
)
from jaxcmr.helpers import find_project_root


def run_comparison(models, run_tag, data_name, fit_dir, target_directory, query_parameters):
    results = []

    for model_name, model_title, fit_tag in models:
        fit_path = os.path.join(
            find_project_root(), fit_dir,
            f"{data_name}_{model_name}_{fit_tag}.json",
        )
        with open(fit_path) as f:
            result = json.load(f)
        if "subject" not in result["fits"]:
            result["fits"]["subject"] = result["subject"]
        if "allow_repeated_recalls" not in result["fixed"]:
            result["fixed"]["allow_repeated_recalls"] = False
            result["fits"]["allow_repeated_recalls"] = [
                False
            ] * len(result["fits"]["subject"])
        result["name"] = model_title
        if "mfc_trace_sensitivity" in result["free"]:
            result["free"]["repetition_orthogonality"] = result["free"][
                "mfc_trace_sensitivity"
            ]
            result["fits"]["repetition_orthogonality"] = result["fits"][
                "mfc_trace_sensitivity"
            ]
            result["free"].pop("mfc_trace_sensitivity")
            result["fits"].pop("mfc_trace_sensitivity")
        results.append(result)

    print(f"Loaded {len(results)} models\n")

    summary = summarize_parameters(
        results, query_parameters, include_std=True, include_ci=True
    )
    table_path = os.path.join(
        find_project_root(), target_directory,
        "results", "tables",
        f"{data_name}_{run_tag}_parameters.md",
    )
    with open(table_path, "w") as f:
        f.write(summary)
    print("Parameter Summary:\n")
    print(summary)

    df_t, df_p = generate_t_p_matrices(results)
    print("\nT-Test P-Value Matrix:\n")
    print(df_p.to_markdown())
    print()

    aic_weights = calculate_aic_weights(results)
    with open(
        os.path.join(
            find_project_root(), target_directory,
            "results", "tables",
            f"{data_name}_{run_tag}_aic_weights.md",
        ),
        "w",
    ) as f:
        f.write(aic_weights.to_markdown())
    print("\nAIC Weights:\n")
    print(aic_weights.to_markdown())
    print()

    df_comparison = winner_comparison_matrix(results)
    with open(
        os.path.join(
            find_project_root(), target_directory,
            "results", "tables",
            f"{data_name}_{run_tag}_winner_ratios.md",
        ),
        "w",
    ) as f:
        f.write(df_comparison.to_markdown().replace(" nan ", "     "))
    print("\nWinner Ratios (AIC):\n")
    print(df_comparison.to_markdown().replace(" nan ", "     "))
    print()

    df_comparison = raw_winner_comparison_matrix(results)
    with open(
        os.path.join(
            find_project_root(), target_directory,
            "results", "tables",
            f"{data_name}_{run_tag}_raw_winner_ratios.md",
        ),
        "w",
    ) as f:
        f.write(df_comparison.to_markdown().replace(" nan ", "     "))
    print("\nRaw Winner Ratios:\n")
    print(df_comparison.to_markdown().replace(" nan ", "     "))
    print()

    mean_delta, ci_halfwidth, _ = pairwise_aic_differences(results)
    delta_aic_table = mean_delta.copy().astype(object)
    for row_label in delta_aic_table.index:
        for col_label in delta_aic_table.columns:
            if row_label == col_label:
                delta_aic_table.loc[row_label, col_label] = ""
                continue
            mean_value = mean_delta.loc[row_label, col_label]
            ci = ci_halfwidth.loc[row_label, col_label]
            if mean_value != mean_value or ci != ci:
                delta_aic_table.loc[row_label, col_label] = ""
            else:
                lower = mean_value - ci
                upper = mean_value + ci
                delta_aic_table.loc[row_label, col_label] = (
                    f"{mean_value:.2f} [{lower:.2f}, {upper:.2f}]"
                )

    with open(
        os.path.join(
            find_project_root(), target_directory,
            "results", "tables",
            f"{data_name}_{run_tag}_delta_aic.md",
        ),
        "w",
    ) as f:
        f.write(delta_aic_table.to_markdown())
    print("\nPairwise ΔAIC (row − column; mean [CI]):\n")
    print(delta_aic_table.to_markdown())
    print()

    # Median ΔAIC (robust to outlier subjects)
    median_delta, median_ci = pairwise_median_aic_differences(results)
    median_table = median_delta.copy().astype(object)
    for row_label in median_table.index:
        for col_label in median_table.columns:
            if row_label == col_label:
                median_table.loc[row_label, col_label] = ""
                continue
            med = median_delta.loc[row_label, col_label]
            ci = median_ci.loc[row_label, col_label]
            if med != med or ci != ci:
                median_table.loc[row_label, col_label] = ""
            else:
                lower = med - ci
                upper = med + ci
                median_table.loc[row_label, col_label] = (
                    f"{med:.2f} [{lower:.2f}, {upper:.2f}]"
                )

    with open(
        os.path.join(
            find_project_root(), target_directory,
            "results", "tables",
            f"{data_name}_{run_tag}_median_delta_aic.md",
        ),
        "w",
    ) as f:
        f.write(median_table.to_markdown())
    print("\nPairwise ΔAIC (row − column; median [bootstrap CI]):\n")
    print(median_table.to_markdown())
    print()

    # Bayesian Model Selection (Stephan et al., 2009)
    bms = bayesian_model_selection(results)
    with open(
        os.path.join(
            find_project_root(), target_directory,
            "results", "tables",
            f"{data_name}_{run_tag}_bms.md",
        ),
        "w",
    ) as f:
        f.write(bms.to_markdown(index=False))
    print("\nBayesian Model Selection:\n")
    print(bms.to_markdown(index=False))
    print()

In [3]:
data_name = "TalmiEEG"
fit_dir = "projects/TalmiEEG/results/fits/"
target_directory = "projects/TalmiEEG/"

_old_tag = "50_set_likelihood_fixed_term_best_of_3"
_ecmr_tag = "20260309_50_set_likelihood_best_of_3"

query_parameters = [
    "encoding_drift_rate",
    "start_drift_rate",
    "recall_drift_rate",
    "shared_support",
    "item_support",
    "learning_rate",
    "primacy_scale",
    "primacy_decay",
    "choice_sensitivity",
    "emotion_scale",
    # "emotion_drift_rate",
    # "lpp_main_scale",
    # "lpp_main_threshold",
    # "lpp_main_exponent",
    "lpp_inter_scale",
    # "lpp_inter_threshold",
    # "lpp_inter_exponent",
    # "lpp_slope",
    # "lpp_threshold",
]

# Manuscript Comparison

In [4]:
import numpy as np

manuscript_models = [
    ("WeirdCMRNoStop", "CMR", _old_tag),
    ("eCMREmotionOnly", "eCMR", _ecmr_tag),
    ("eCMREmotionTimesLPP", "LPP-eCMR", _ecmr_tag),
]

# Load models (same logic as run_comparison)
results = []
for model_name, model_title, fit_tag in manuscript_models:
    fit_path = os.path.join(
        find_project_root(), fit_dir,
        f"{data_name}_{model_name}_{fit_tag}.json",
    )
    with open(fit_path) as f:
        result = json.load(f)
    if "subject" not in result["fits"]:
        result["fits"]["subject"] = result["subject"]
    if "allow_repeated_recalls" not in result["fixed"]:
        result["fixed"]["allow_repeated_recalls"] = False
        result["fits"]["allow_repeated_recalls"] = [
            False
        ] * len(result["fits"]["subject"])
    result["name"] = model_title
    if "mfc_trace_sensitivity" in result["free"]:
        result["free"]["repetition_orthogonality"] = result["free"][
            "mfc_trace_sensitivity"
        ]
        result["fits"]["repetition_orthogonality"] = result["fits"][
            "mfc_trace_sensitivity"
        ]
        result["free"].pop("mfc_trace_sensitivity")
        result["fits"].pop("mfc_trace_sensitivity")
    results.append(result)

# Floor LPP-eCMR fitness at eCMR fitness (nesting fix).
# LPP-eCMR (k=11) nests eCMR (k=10): setting lpp_inter_scale ≈ 0
# recovers eCMR exactly. Any subject where LPP-eCMR has worse NLL
# than eCMR is an optimizer failure, not a model property.
ecmr_fitness = np.array(results[1]["fitness"])
lpp_fitness = np.array(results[2]["fitness"])
n_floored = int(np.sum(lpp_fitness > ecmr_fitness))
print(f"Floored {n_floored}/{len(lpp_fitness)} subjects "
      f"(LPP-eCMR NLL > eCMR NLL due to optimizer noise)\n")
results[2]["fitness"] = np.minimum(lpp_fitness, ecmr_fitness).tolist()

# Run all analyses
run_tag = "Manuscript_Comparison"

print(f"Loaded {len(results)} models\n")

summary = summarize_parameters(
    results, query_parameters, include_std=True, include_ci=True
)
table_path = os.path.join(
    find_project_root(), target_directory,
    "results", "tables",
    f"{data_name}_{run_tag}_parameters.md",
)
with open(table_path, "w") as f:
    f.write(summary)
print("Parameter Summary:\n")
print(summary)

df_t, df_p = generate_t_p_matrices(results)
print("\nT-Test P-Value Matrix:\n")
print(df_p.to_markdown())
print()

aic_weights = calculate_aic_weights(results)
with open(
    os.path.join(
        find_project_root(), target_directory,
        "results", "tables",
        f"{data_name}_{run_tag}_aic_weights.md",
    ),
    "w",
) as f:
    f.write(aic_weights.to_markdown())
print("\nAIC Weights:\n")
print(aic_weights.to_markdown())
print()

df_comparison = winner_comparison_matrix(results)
with open(
    os.path.join(
        find_project_root(), target_directory,
        "results", "tables",
        f"{data_name}_{run_tag}_winner_ratios.md",
    ),
    "w",
) as f:
    f.write(df_comparison.to_markdown().replace(" nan ", "     "))
print("\nWinner Ratios (AIC):\n")
print(df_comparison.to_markdown().replace(" nan ", "     "))
print()

df_comparison = raw_winner_comparison_matrix(results)
with open(
    os.path.join(
        find_project_root(), target_directory,
        "results", "tables",
        f"{data_name}_{run_tag}_raw_winner_ratios.md",
    ),
    "w",
) as f:
    f.write(df_comparison.to_markdown().replace(" nan ", "     "))
print("\nRaw Winner Ratios:\n")
print(df_comparison.to_markdown().replace(" nan ", "     "))
print()

mean_delta, ci_halfwidth, _ = pairwise_aic_differences(results)
delta_aic_table = mean_delta.copy().astype(object)
for row_label in delta_aic_table.index:
    for col_label in delta_aic_table.columns:
        if row_label == col_label:
            delta_aic_table.loc[row_label, col_label] = ""
            continue
        mean_value = mean_delta.loc[row_label, col_label]
        ci = ci_halfwidth.loc[row_label, col_label]
        if mean_value != mean_value or ci != ci:
            delta_aic_table.loc[row_label, col_label] = ""
        else:
            lower = mean_value - ci
            upper = mean_value + ci
            delta_aic_table.loc[row_label, col_label] = (
                f"{mean_value:.2f} [{lower:.2f}, {upper:.2f}]"
            )

with open(
    os.path.join(
        find_project_root(), target_directory,
        "results", "tables",
        f"{data_name}_{run_tag}_delta_aic.md",
    ),
    "w",
) as f:
    f.write(delta_aic_table.to_markdown())
print("\nPairwise ΔAIC (row − column; mean [CI]):\n")
print(delta_aic_table.to_markdown())
print()

median_delta, median_ci = pairwise_median_aic_differences(results)
median_table = median_delta.copy().astype(object)
for row_label in median_table.index:
    for col_label in median_table.columns:
        if row_label == col_label:
            median_table.loc[row_label, col_label] = ""
            continue
        med = median_delta.loc[row_label, col_label]
        ci = median_ci.loc[row_label, col_label]
        if med != med or ci != ci:
            median_table.loc[row_label, col_label] = ""
        else:
            lower = med - ci
            upper = med + ci
            median_table.loc[row_label, col_label] = (
                f"{med:.2f} [{lower:.2f}, {upper:.2f}]"
            )

with open(
    os.path.join(
        find_project_root(), target_directory,
        "results", "tables",
        f"{data_name}_{run_tag}_median_delta_aic.md",
    ),
    "w",
) as f:
    f.write(median_table.to_markdown())
print("\nPairwise ΔAIC (row − column; median [bootstrap CI]):\n")
print(median_table.to_markdown())
print()

bms = bayesian_model_selection(results)
with open(
    os.path.join(
        find_project_root(), target_directory,
        "results", "tables",
        f"{data_name}_{run_tag}_bms.md",
    ),
    "w",
) as f:
    f.write(bms.to_markdown(index=False))
print("\nBayesian Model Selection:\n")
print(bms.to_markdown(index=False))
print()

Floored 10/38 subjects (LPP-eCMR NLL > eCMR NLL due to optimizer noise)

Loaded 3 models

Parameter Summary:

| Parameter | Statistic | CMR | eCMR | LPP-eCMR |
|---|---|---|---|---|
| fitness | mean | 215.01 +/- 15.63 | 211.98 +/- 16.31 | 210.83 +/- 16.46 |
|  | std | 46.91 | 48.98 | 49.42 |
|  | min | 130.91 | 114.22 | 113.86 |
|  | max | 318.62 | 315.97 | 315.97 |
| encoding drift rate | mean | 0.66 +/- 0.11 | 0.65 +/- 0.10 | 0.73 +/- 0.09 |
|  | std | 0.32 | 0.30 | 0.27 |
|  | min | 0.13 | 0.06 | 0.00 |
|  | max | 1.00 | 1.00 | 1.00 |
| start drift rate | mean | 0.36 +/- 0.12 | 0.42 +/- 0.13 | 0.44 +/- 0.13 |
|  | std | 0.35 | 0.39 | 0.39 |
|  | min | 0.00 | 0.00 | 0.00 |
|  | max | 1.00 | 1.00 | 1.00 |
| recall drift rate | mean | 0.57 +/- 0.12 | 0.64 +/- 0.10 | 0.64 +/- 0.10 |
|  | std | 0.35 | 0.29 | 0.30 |
|  | min | 0.00 | 0.00 | 0.01 |
|  | max | 1.00 | 1.00 | 1.00 |
| shared support | mean | 37.15 +/- 10.16 | 42.03 +/- 11.58 | 43.83 +/- 11.87 |
|  | std | 30.51 | 34.76 | 35.6

# Extended Comparison

In [5]:
extended_models = [
    ("WeirdCMRNoStop", "CMR", _old_tag),
    ("EEGEmotionOnly", "Emotion", _old_tag),
    ("EEGMainEffects", "Emo+LPP", _old_tag),
    ("EEGMainEffectsPlusInteraction", "Emo+LPP+Int", _old_tag),
    ("eCMRInteraction", "eCMR Emo+LPP+Int", _ecmr_tag),
    ("eCMREmotionTimesLPP", "eCMR Emo×LPP", _ecmr_tag),
    ("eCMREmotionBroad", "eCMR Emotion Broad", _ecmr_tag),
    ("eCMREmotionTimesLPPBroad", "eCMR Emo×LPP Broad", _ecmr_tag),
]

run_comparison(extended_models, "Extended_Comparison", data_name, fit_dir, target_directory, query_parameters)

Loaded 8 models

Parameter Summary:

| Parameter | Statistic | CMR | Emotion | Emo+LPP | Emo+LPP+Int | eCMR Emo+LPP+Int | eCMR Emo×LPP | eCMR Emotion Broad | eCMR Emo×LPP Broad |
|---|---|---|---|---|---|---|---|---|---|
| fitness | mean | 215.01 +/- 15.63 | 211.74 +/- 16.38 | 210.26 +/- 16.34 | 209.65 +/- 16.24 | 209.92 +/- 16.22 | 211.06 +/- 16.49 | 211.97 +/- 16.36 | 210.95 +/- 16.50 |
|  | std | 46.91 | 49.17 | 49.05 | 48.75 | 48.69 | 49.50 | 49.12 | 49.53 |
|  | min | 130.91 | 113.81 | 113.40 | 113.79 | 112.84 | 113.86 | 116.18 | 114.44 |
|  | max | 318.62 | 318.67 | 315.56 | 316.06 | 313.45 | 318.56 | 318.57 | 318.22 |
| encoding drift rate | mean | 0.66 +/- 0.11 | 0.62 +/- 0.11 | 0.61 +/- 0.11 | 0.65 +/- 0.11 | 0.75 +/- 0.08 | 0.73 +/- 0.09 | 0.66 +/- 0.11 | 0.70 +/- 0.10 |
|  | std | 0.32 | 0.32 | 0.33 | 0.33 | 0.24 | 0.27 | 0.32 | 0.30 |
|  | min | 0.13 | 0.00 | 0.00 | 0.00 | 0.28 | 0.00 | 0.00 | 0.00 |
|  | max | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 |
| start

# Full Comparison (Supplement)

In [6]:
full_models = [
    ("WeirdCMRNoStop", "CMR", _old_tag),
    ("EEGEmotionOnly", "Emotion", _old_tag),
    ("EEGLPPOnly", "LPP", _old_tag),
    ("EEGLPPExponentOnly", "LPP^exp", _old_tag),
    ("EEGMainEffects", "Emo+LPP", _old_tag),
    ("EEGMainEffectsPlusInteraction", "Emo+LPP+Int", _old_tag),
    ("EEGTwoLayerMainEffects", "2L Emo+LPP", _old_tag),
    ("EEGTwoLayerInteraction", "2L Emo+LPP+Int", _old_tag),
    ("eCMREmotionOnly", "eCMR Emotion", _ecmr_tag),
    ("eCMREmotionTimesLPP", "eCMR Emo×LPP", _ecmr_tag),
    ("eCMRMainEffects", "eCMR Emo+LPP", _ecmr_tag),
    ("eCMRInteraction", "eCMR Emo+LPP+Int", _ecmr_tag),
    ("eCMREmotionBroad", "eCMR Emotion Broad", _ecmr_tag),
    ("eCMREmotionTimesLPPBroad", "eCMR Emo×LPP Broad", _ecmr_tag),
]

run_comparison(full_models, "Full_Comparison", data_name, fit_dir, target_directory, query_parameters)

Loaded 14 models

Parameter Summary:

| Parameter | Statistic | CMR | Emotion | LPP | LPP^exp | Emo+LPP | Emo+LPP+Int | 2L Emo+LPP | 2L Emo+LPP+Int | eCMR Emotion | eCMR Emo×LPP | eCMR Emo+LPP | eCMR Emo+LPP+Int | eCMR Emotion Broad | eCMR Emo×LPP Broad |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| fitness | mean | 215.01 +/- 15.63 | 211.74 +/- 16.38 | 213.07 +/- 15.71 | 213.28 +/- 15.89 | 210.26 +/- 16.34 | 209.65 +/- 16.24 | 211.77 +/- 16.33 | 211.46 +/- 16.27 | 211.98 +/- 16.31 | 211.06 +/- 16.49 | 210.45 +/- 16.29 | 209.92 +/- 16.22 | 211.97 +/- 16.36 | 210.95 +/- 16.50 |
|  | std | 46.91 | 49.17 | 47.18 | 47.70 | 49.05 | 48.75 | 49.01 | 48.85 | 48.98 | 49.50 | 48.91 | 48.69 | 49.12 | 49.53 |
|  | min | 130.91 | 113.81 | 128.54 | 128.18 | 113.40 | 113.79 | 116.37 | 116.36 | 114.22 | 113.86 | 114.77 | 112.84 | 116.18 | 114.44 |
|  | max | 318.62 | 318.67 | 315.52 | 318.26 | 315.56 | 316.06 | 318.65 | 318.49 | 315.97 | 318.56 | 315.52 | 313.45 | 318.57 | 318.2